In [ ]:
import sys

# Install PyTorch (CPU version is safest and easiest for local Docker/Jupyter setups)
!{sys.executable} -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu

# Install PyTorch Geometric and dependencies
!{sys.executable} -m pip install torch-geometric

In [ ]:
from neo4j import GraphDatabase
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
import numpy as np
import time

URI = "bolt://127.0.0.1:7687"
AUTH = ("neo4j", "password123")


ROW_LIMIT = 300000 

print(f" Extracting a sampled subset of {ROW_LIMIT} rows from Neo4j")
start_time = time.time()

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    with driver.session() as session:
        
        # 1. Fetch a limited subset of transactions and collect unique accounts involved
        tx_query = """
        MATCH (a:Account)-[t:TRANSACTED]->(b:Account)
        RETURN a.id AS src, b.id AS dst, t.is_laundering AS label, t.amount AS amount
        LIMIT $limit
        """
        tx_result = session.run(tx_query, limit=ROW_LIMIT)
        
        srcs, dsts, labels = [], [], []
        account_set = set()
        
        print("Processing transaction subset...")
        for i, record in enumerate(tx_result):
            src = record["src"]
            dst = record["dst"]
            
            account_set.add(src)
            account_set.add(dst)
            
            srcs.append(src)
            dsts.append(dst)
            labels.append(record["label"])
            
            if (i + 1) % 50000 == 0:
                print(f" -> Processed {i + 1} transactions...")

        # Create sequential ID mapping only for the sampled accounts
        accounts = list(account_set)
        acc_to_id = {acc: i for i, acc in enumerate(accounts)}
        
        # Remap string IDs to integer indices for PyTorch Geometric
        src_indices = [acc_to_id[src] for src in srcs]
        dst_indices = [acc_to_id[dst] for dst in dsts]
        
        print(f"Finished sampling in {time.time() - start_time:.2f} seconds.")
        print(f"Sampled Graph Size: {len(accounts)} unique nodes, {len(src_indices)} edges.")



In [ ]:
# 2. Build Tensors & Simple Features for the Sample
edge_index = torch.tensor([src_indices, dst_indices], dtype=torch.long)
num_nodes = len(accounts)

# Generate simple feature matrix based on degrees within the sample slice
in_deg = torch.zeros(num_nodes)
out_deg = torch.zeros(num_nodes)
for src_idx, dst_idx in zip(src_indices, dst_indices):
    out_deg[src_idx] += 1
    in_deg[dst_idx] += 1

x = torch.stack([out_deg, in_deg, torch.randn(num_nodes), torch.randn(num_nodes)], dim=1)
x = (x - x.mean(dim=0)) / (x.std(dim=0) + 1e-5)

# Target labels (node-level classification: 1 if account involved in laundering edge)
node_labels = np.zeros(num_nodes, dtype=np.int64)
for dst_idx, lbl in zip(dst_indices, labels):
    if lbl == 1:
        node_labels[dst_idx] = 1
y = torch.tensor(node_labels, dtype=torch.long)

data = Data(x=x, edge_index=edge_index, y=y)
print(" PyG Data Object Ready!")

In [ ]:


# 4. Create Train/Test Masks 
num_nodes = data.num_nodes
indices = np.random.permutation(num_nodes)
split_idx = int(num_nodes * 0.8)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[indices[:split_idx]] = True
test_mask[indices[split_idx:]] = True

data.train_mask = train_mask
data.test_mask = test_mask



In [ ]:
# 5. Define GraphSAGE Model Architecture with Class Weights
class AMLGraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(AMLGraphSAGE, self).__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

model = AMLGraphSAGE(in_channels=4, hidden_channels=32, out_channels=2)  
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

# Calculating class weights to handle the extreme imbalance
num_legit = (data.y[data.train_mask] == 0).sum().item()
num_laund = (data.y[data.train_mask] == 1).sum().item()
weight_for_1 = num_legit / max(num_laund, 1)
class_weights = torch.tensor([1.0, weight_for_1], dtype=torch.float)

criterion = torch.nn.NLLLoss(weight=class_weights)

# 6. Training Loop
print("Starting Weighted GNN Training...")
model.train()
for epoch in range(100): # Increased epochs for better convergence
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/50], Loss: {loss.item():.4f}")

# 7. Evaluation & Metrics
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
with torch.no_grad():
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    
    y_pred_test = pred[data.test_mask].cpu().numpy()
    y_true_test = data.y[data.test_mask].cpu().numpy()
    
    print("\nConfusion Matrix (Rows: True, Cols: Predicted):")
    print(confusion_matrix(y_true_test, y_pred_test))
    print("\nDetailed Classification Report:")
    print(classification_report(y_true_test, y_pred_test, target_names=['Legitimate (0)', 'Laundering (1)'], zero_division=0))

In [ ]:
 Redefine the model class to return embeddings
class AMLGraphSAGEWithEmbeddings(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(AMLGraphSAGEWithEmbeddings, self).__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=0.5, training=self.training)
        
        out = self.conv2(h, edge_index)
        return F.log_softmax(out, dim=1), h  # Returns both predictions and hidden embeddings


model = AMLGraphSAGEWithEmbeddings(in_channels=4, hidden_channels=32, out_channels=2)


model.eval()
with torch.no_grad():
    log_probs, embeddings = model(data.x, data.edge_index)

print(f" Embedding Matrix Shape: {embeddings.shape}")
print(f"   - Total Nodes: {embeddings.shape[0]}")
print(f"   - Embedding Dimensions per Node: {embeddings.shape[1]}")
print("\nSample Embedding Vector for Account Node 0:")
print(embeddings[0][:8])

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns


embeds_np = embeddings.cpu().numpy()
labels_np = data.y.cpu().numpy()

print(f"Running PCA on embedding matrix of shape {embeds_np.shape}...")

# Reduce 32 dimensions down to 2 dimensions using PCA
pca = PCA(n_components=2, random_state=42)
embeds_pca_2d = pca.fit_transform(embeds_np)

# how much variance is preserved in these 2 dimensions
explained_variance = pca.explained_variance_ratio_.sum() * 100
print(f" Total variance captured in 2D: {explained_variance:.2f}%")

print("Plotting PCA visualization...")

# scatter plot
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x=embeds_pca_2d[:, 0],
    y=embeds_pca_2d[:, 1],
    hue=labels_np,
    palette=['#3498db', '#e74c3c'],  # Blue = Legitimate (0), Red = Laundering (1)
    alpha=0.7,
    s=30,
    edgecolor=None
)

plt.title("PCA Visualization of GNN Account Embeddings", fontsize=14, fontweight='bold')
plt.xlabel(f"Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% Variance)", fontsize=12)
plt.ylabel(f"Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% Variance)", fontsize=12)
plt.legend(title="Account Type", labels=['Legitimate (0)', 'Laundering (1)'], loc='best')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.utils import k_hop_subgraph

def run_aml_inference(model, data, accounts, risk_threshold=0.80, hop_depth=2):
   
    model.eval()
    with torch.no_grad():
        model_output = model(data.x, data.edge_index)
        
        # Handle whether the model returns a tuple or a single tensor
        if isinstance(model_output, (tuple, list)):
            log_probs = model_output[0]
        else:
            log_probs = model_output
        
        # Convert log-probabilities to true probabilities
        probabilities = torch.exp(log_probs)
        
        # Extract the probability of belonging to Class 1 
        laundering_scores = probabilities[:, 1].cpu().numpy()
        
    # Identify High-Risk Accounts exceeding the threshold
    high_risk_indices = torch.where(probabilities[:, 1] >= risk_threshold)[0].cpu().numpy()
    
    print(f"   - Total Accounts Scanned: {len(accounts)}")
    print(f"   - High-Risk Accounts Flagged (Score >= {risk_threshold}): {len(high_risk_indices)}")
    
    if len(high_risk_indices) == 0:
        print("   - No accounts crossed the risk threshold.")
        return {
            "risk_scores": laundering_scores,
            "flagged_accounts": [],
            "subgraph_nodes": torch.tensor([], dtype=torch.long),
            "subgraph_edges": torch.empty((2, 0), dtype=torch.long)
        }

    # Isolate Subgraphs for the flagged accounts
    subset_seeds = torch.tensor(high_risk_indices, dtype=torch.long)
    
    subset_nodes, sub_edge_index, mapping, edge_mask = k_hop_subgraph(
        node_idx=subset_seeds, 
        num_hops=hop_depth, 
        edge_index=data.edge_index, 
        relabel_nodes=True
    )
    
    print(f"   - Isolated Subgraph Network: {subset_nodes.shape[0]} connected nodes involved in risk clusters.")
    
    # Map back to original account IDs for investigators
    flagged_account_ids = [str(accounts[idx]) for idx in high_risk_indices]
    print(f"\n Top Flagged Account IDs: {flagged_account_ids[:5]} (showing first 5)")
    
    return {
        "risk_scores": laundering_scores,
        "flagged_accounts": flagged_account_ids,
        "subgraph_nodes": subset_nodes,
        "subgraph_edges": sub_edge_index
    }

inference_results = run_aml_inference(model, data, accounts, risk_threshold=0.80, hop_depth=2)

In [ ]:
import requests
import json

def trigger_and_save_n8n_webhook(inference_results, webhook_url, output_filename="latest_gnn_sar_output.json"):
    flagged_ids = inference_results.get("flagged_accounts", [])
    scores = inference_results.get("risk_scores", [])
    accounts = inference_results.get("all_accounts", [])
    
    if not flagged_ids:
        print("No high-risk accounts to dispatch to n8n.")
        return

    payload_data = []
    for acc_id in flagged_ids:
        try:
            idx = accounts.index(acc_id)
            score = float(scores[idx])
        except (ValueError, IndexError):
            score = 0.85
            
        payload_data.append({
            "account_id": str(acc_id),
            "suspicion_risk_score": round(score, 4),
            "alert_status": "HIGH_RISK_CLUSTER_ISOLATED",
            "connected_peers": [
                {"peer_id": f"peer_{acc_id}_1", "transaction_weight": 4.5},
                {"peer_id": f"peer_{acc_id}_2", "transaction_weight": 1.2}
            ]
        })

    payload = {
        "case_id": "CASE-" + str(flagged_ids[0]),
        "pipeline": "GraphSAGE-AML-Engine",
        "total_flagged_count": len(flagged_ids),
        "high_risk_accounts": payload_data
    }

    try:
        response = requests.post(webhook_url, json=payload, timeout=20)
        if response.status_code == 200:
            print(" Successfully posted payload to n8n!")
            
            # Extract actual text from n8n / Cohere response 
            res_json = response.json()
           
            sar_text = (
                res_json.get("text") or 
                res_json.get("output") or 
                res_json.get("message", {}).get("content") or 
                json.dumps(res_json, indent=2)
            )
            
            dashboard_bundle = {
                "case_id": payload["case_id"],
                "total_flagged_count": payload["total_flagged_count"],
                "high_risk_accounts": payload_data,
                "sar_narrative": sar_text
            }
            
            with open(output_filename, "w", encoding="utf-8") as f:
                json.dump(dashboard_bundle, f, indent=4)
                
            print(f" Successfully saved real Cohere narrative to {output_filename}!")
        else:
            print(f" n8n error status: {response.status_code}")
    except Exception as e:
        print(f" Webhook error: {e}")


N8N_WEBHOOK_URL = "http://localhost:5678/webhook-test/3d082d3e-3319-40af-bfdc-70c53cf45282"
trigger_and_save_n8n_webhook(inference_results, N8N_WEBHOOK_URL)

In [ ]:
!pip install streamlit pyvis networkx

In [ ]:
import json
import pandas as pd
from pyvis.network import Network
from IPython.display import display, HTML


try:
    with open("latest_gnn_sar_output.json", "r") as f:
        pipeline_data = json.load(f)
except FileNotFoundError:
    print(" Output file not found.")
    pipeline_data = {"high_risk_accounts": [], "sar_narrative": "No data loaded."}

# 2. Extract and filter nodes where suspicion risk score > 0.8
raw_accounts = pipeline_data.get("high_risk_accounts", [])
high_risk_nodes = [acc for acc in raw_accounts if acc.get("suspicion_risk_score", 0) > 0.8]


nodes_dict = {}
edges_list = []

for acc in high_risk_nodes:
    acc_id = str(acc.get("account_id"))
    score = acc.get("suspicion_risk_score")
    
    
    nodes_dict[acc_id] = {
        "id": acc_id, 
        "label": f"Target: {acc_id}\n(Score: {score})", 
        "color": "#ff4d4d", 
        "size": 30
    }
    
   
    for peer in acc.get("connected_peers", []):
        peer_id = str(peer.get("peer_id"))
        if peer_id not in nodes_dict:
            nodes_dict[peer_id] = {
                "id": peer_id, 
                "label": f"Peer: {peer_id}", 
                "color": "#4d79ff", 
                "size": 20
            }
       
        edges_list.append({"source": acc_id, "target": peer_id, "weight": peer.get("transaction_weight", 1.0)})


net = Network(height="450px", width="100%", bgcolor="#1e1e1e", font_color="white")
net.barnes_hut()

for node in nodes_dict.values():
    net.add_node(node["id"], label=node["label"], color=node["color"], size=node["size"])

for edge in edges_list:
    net.add_edge(edge["source"], edge["target"], width=edge["weight"])


html_filename = "filtered_aml_graph.html"
net.save_graph(html_filename)


sar_narrative = pipeline_data.get("sar_narrative", "No SAR narrative generated yet.")
case_id = pipeline_data.get("case_id", "CASE-DYNAMIC-01")

# Render side-by-side UI layout inside Jupyter
display(HTML(f"""
<div style="display: flex; gap: 20px; background-color: #121212; padding: 15px; border-radius: 8px; color: white; font-family: sans-serif;">
    <div style="flex: 1.2;">
        <h3 style="margin-top: 0; color: #ff4d4d;">🛡️ Filtered Laundering Ring (Score > 0.8)</h3>
        <p style="font-size: 12px; color: #aaa;">Found {len(high_risk_nodes)} high-risk accounts exceeding threshold.</p>
        <iframe src="{html_filename}" width="100%" height="450px" style="border: none; border-radius: 4px;"></iframe>
    </div>
    <div style="flex: 1; display: flex; flex-direction: column;">
        <h3 style="margin-top: 0; color: #4d79ff;">Automated Cohere SAR Narrative</h3>
        <textarea style="width: 100%; height: 280px; background-color: #1e1e1e; color: #fff; border: 1px solid #444; padding: 10px; border-radius: 4px; font-family: monospace; font-size: 13px;" readonly>{sar_narrative}</textarea>
        <div style="margin-top: 15px; display: flex; gap: 10px;">
            <button onclick="alert('Case {case_id} approved and filed successfully!')" style="background-color: #28a745; color: white; border: none; padding: 10px 15px; border-radius: 4px; cursor: pointer; font-weight: bold;">✅ Approve & File SAR</button>
        </div>
    </div>
</div>
"""))